# Module 09: Agents & Multi-Agent Systems

**Companion notebook for**: `09-agents.html` -- CareerAlign GenAI Course

## Learning Objectives

By the end of this notebook you will be able to:

- Understand and implement the **ReAct pattern** (Reason + Act) from scratch
- Define **tools** with JSON schemas and bind them to LLM calls
- Build **LangGraph state machines** with nodes, edges, and conditional routing
- Implement a **multi-agent supervisor pattern** where a manager delegates to specialists
- Add **human-in-the-loop** approval gates using `interrupt_before`
- Persist **agent memory and state** with LangGraph checkpointers
- Apply **safety and reliability** patterns: budgets, retries, logging
- Build an **end-to-end research assistant** agent

## Prerequisites

- OpenAI API key set as `OPENAI_API_KEY` environment variable
- Familiarity with Python type hints and `TypedDict`

---
## Setup & Dependencies

In [ ]:
%pip install -q openai langgraph langchain-core langchain-openai

In [ ]:
import os
import json
import time
import logging
from typing import TypedDict, Annotated, Literal
from operator import add
from functools import wraps

from openai import OpenAI

# Verify API key
assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY environment variable"

client = OpenAI()
print("All imports successful. OpenAI API key found.")

---
## Section 1: The ReAct Pattern

The **ReAct** pattern (Reasoning + Acting) is the foundational framework for LLM agents.
It mirrors how humans solve problems:

1. **Reason** -- Think about the problem and plan the next step
2. **Act** -- Execute a tool or take an action
3. **Observe** -- Examine the result
4. **Repeat** until the task is complete

Unlike traditional programming where every step is specified in advance, a ReAct agent
figures out the steps *at runtime* based on what it discovers along the way. If a search
returns ambiguous results, the agent can rephrase and search again. If an API call fails,
it can try an alternative approach.

The loop typically runs for 3--10 iterations. A maximum step cap prevents runaway loops
that burn tokens and API costs.

### 1.1 Simple ReAct Agent from Scratch

Below we build a minimal ReAct agent using the OpenAI function-calling API.
No frameworks -- just the core loop.

In [ ]:
# --- Tool Definitions (OpenAI function-calling format) ---
tools = [
    {
        "type": "function",
        "function": {
            "name": "web_search",
            "description": "Search the web for current information. Use when you need facts you don't know.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The search query"
                    }
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Perform mathematical calculations. Input is a math expression.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "Math expression to evaluate, e.g. '2 + 3 * 4'"
                    }
                },
                "required": ["expression"]
            }
        }
    }
]

print("Defined 2 tools: web_search, calculator")
print(json.dumps(tools[0]["function"], indent=2))

In [ ]:
# --- Tool Implementations ---
def execute_tool(name: str, args: dict) -> str:
    """Dispatch a tool call to the appropriate implementation."""
    if name == "web_search":
        # In production, use a real search API (Tavily, Serper, etc.)
        query = args["query"]
        # Simulated results for demonstration
        simulated = {
            "population": "The current US population is approximately 335 million (2024 estimate).",
            "apple founded": "Apple was founded in Cupertino, California on April 1, 1976.",
            "iphone announced": "The iPhone was announced by Steve Jobs on January 9, 2007.",
        }
        for keyword, result in simulated.items():
            if keyword in query.lower():
                return result
        return f"Search results for '{query}': No specific results found. The query may need refinement."

    elif name == "calculator":
        try:
            # WARNING: eval() is used here for simplicity.
            # In production, use a safe math parser like numexpr or asteval.
            result = eval(args["expression"])
            return str(result)
        except Exception as e:
            return f"Calculation error: {e}"

    return f"Unknown tool: {name}"


# Quick test
print(execute_tool("calculator", {"expression": "335_000_000 * 0.15"}))
print(execute_tool("web_search", {"query": "US population estimate"}))

In [ ]:
# --- The ReAct Agent Loop ---
def run_react_agent(user_message: str, max_steps: int = 10) -> str:
    """
    Run a ReAct agent loop.

    The agent alternates between:
      - Reasoning (LLM decides what to do)
      - Acting    (tool execution)
      - Observing (result fed back to LLM)

    until the LLM produces a final text answer (no tool calls).
    """
    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful assistant with access to tools. "
                "Think step by step. Use tools when you need information "
                "you don't have. When you have enough information, provide "
                "a final answer to the user."
            ),
        },
        {"role": "user", "content": user_message},
    ]

    for step in range(1, max_steps + 1):
        print(f"\n--- Step {step} ---")

        # REASON: LLM decides what to do next
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=tools,
            tool_choice="auto",
        )

        msg = response.choices[0].message
        messages.append(msg)

        # If no tool calls, the agent is done -- return the final answer
        if not msg.tool_calls:
            print("Agent finished (no more tool calls).")
            return msg.content

        # ACT: Execute each tool call
        for tool_call in msg.tool_calls:
            fn_name = tool_call.function.name
            fn_args = json.loads(tool_call.function.arguments)
            print(f"  Tool call: {fn_name}({fn_args})")

            result = execute_tool(fn_name, fn_args)
            print(f"  Result: {result}")

            # OBSERVE: Feed the result back to the LLM
            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": result,
                }
            )

    return "Agent reached maximum steps without completing the task."


# Run the agent
answer = run_react_agent("What is 15% of the current US population?")
print(f"\nFinal Answer: {answer}")

**Key takeaways from the ReAct loop:**

- The LLM *decides* which tool to call and with what arguments (reasoning).
- Your code *executes* the tool and feeds the result back (acting + observing).
- The loop continues until the LLM responds with plain text (no tool calls) or the step limit is reached.
- This is the same pattern that powers ChatGPT plugins, Claude tool use, and most agent frameworks.

---
## Section 2: Tool Definition and Binding

Tools are the "arms and legs" of an agent. Without them, an agent can only generate text.
With tools, it can search the web, query databases, execute code, send emails, and more.

The mechanism is **function calling**: the LLM generates structured JSON specifying which
function to call and with what arguments. Your code executes the function and feeds the
result back.

**Tool description quality is critical.** Precise descriptions help the LLM select the
right tool. Include:
1. What the tool does
2. When to use it
3. Parameter constraints and examples

Most production agents use 3--10 tools. Too many tools confuse the model.

In [ ]:
# --- Production-quality tool definitions with langchain-core ---
from langchain_core.tools import tool


@tool
def search_knowledge_base(query: str) -> str:
    """Search the company knowledge base for relevant documents.
    Use when the user asks about internal policies, procedures, or
    company-specific information. The query should be a natural
    language search phrase.
    """
    # Simulated -- in production, call your vector store or search API
    return f"Knowledge base results for '{query}': [Company policy on remote work: ...]"


@tool
def query_database(sql: str) -> str:
    """Execute a read-only SQL query against the application database.
    Returns results as JSON. Only SELECT queries are allowed.
    Use for customer lookups, order history, and product searches.
    """
    if not sql.strip().upper().startswith("SELECT"):
        return "Error: Only SELECT queries are permitted."
    # Simulated
    return json.dumps([{"id": 1, "name": "Widget", "price": 29.99}])


@tool
def send_notification(recipient: str, message: str) -> str:
    """Send a notification message to a user. Requires human approval
    for messages to external recipients. Use for alerts and follow-ups.
    """
    return f"Notification queued for {recipient}: '{message[:50]}...'"


# Inspect the auto-generated schema
all_tools = [search_knowledge_base, query_database, send_notification]
for t in all_tools:
    print(f"{t.name}: {t.description[:60]}...")
print(f"\nSchema for 'query_database':")
print(json.dumps(query_database.args_schema.model_json_schema(), indent=2))

In [ ]:
# --- Binding tools to a chat model with langchain-openai ---
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# bind_tools attaches the tool schemas to every LLM call
llm_with_tools = llm.bind_tools(all_tools)

# The LLM now knows about our tools and can request them
response = llm_with_tools.invoke("Look up the remote work policy for me.")
print("Content:", response.content)
print("Tool calls:", response.tool_calls)

---
## Section 3: LangGraph State Machine Basics

While the basic ReAct loop works for straightforward tasks, complex workflows need
more structure. Consider a customer support agent that must:

1. Identify the customer
2. Look up order history
3. Determine issue type
4. Route to the right resolution (refund, replacement, tech support)
5. Execute the resolution with approval

This is a **directed graph** with conditional branches, loops, and approval gates.

**LangGraph** models these workflows as state machines:
- **Nodes** = functions that do work
- **Edges** = connections between nodes
- **Conditional edges** = branches based on state
- **State** = a `TypedDict` that flows through the graph

LangGraph's key advantage is **cycles** -- the ability to loop back to earlier nodes.
This is impossible with simple sequential chains but natural in a graph.

In [ ]:
from langgraph.graph import StateGraph, END


# --- Step 1: Define the state ---
class SimpleState(TypedDict):
    """State that flows through our graph."""
    input_text: str
    word_count: int
    classification: str
    output: str


# --- Step 2: Define node functions ---
# Each node receives the full state and returns a partial update.

def count_words(state: SimpleState) -> dict:
    """Count the words in the input text."""
    count = len(state["input_text"].split())
    print(f"  [count_words] {count} words")
    return {"word_count": count}


def classify_length(state: SimpleState) -> dict:
    """Classify the text as short, medium, or long."""
    count = state["word_count"]
    if count < 10:
        label = "short"
    elif count < 50:
        label = "medium"
    else:
        label = "long"
    print(f"  [classify_length] -> {label}")
    return {"classification": label}


def summarize(state: SimpleState) -> dict:
    """Produce a summary output."""
    output = (
        f"Text is {state['classification']} ({state['word_count']} words). "
        f"First 30 chars: '{state['input_text'][:30]}...'"
    )
    print(f"  [summarize] done")
    return {"output": output}


# --- Step 3: Build the graph ---
graph = StateGraph(SimpleState)

# Add nodes
graph.add_node("count_words", count_words)
graph.add_node("classify", classify_length)
graph.add_node("summarize", summarize)

# Set entry point and add edges
graph.set_entry_point("count_words")
graph.add_edge("count_words", "classify")
graph.add_edge("classify", "summarize")
graph.add_edge("summarize", END)

# Compile
app = graph.compile()

# --- Step 4: Run ---
result = app.invoke({
    "input_text": "LangGraph makes it easy to build complex agent workflows as state machines.",
    "word_count": 0,
    "classification": "",
    "output": "",
})

print(f"\nResult: {result['output']}")

### 3.1 Conditional Routing in LangGraph

Conditional edges let the graph take different paths based on the current state.
A routing function inspects the state and returns a string key that maps to the
next node. This enables branching, looping, and early termination.

In [ ]:
# --- Conditional routing example ---

class RouterState(TypedDict):
    query: str
    category: str
    response: str


def categorize(state: RouterState) -> dict:
    """Use the LLM to categorize the user query."""
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": (
                    "Classify the query into exactly one category: "
                    "'technical', 'billing', or 'general'. "
                    "Return only the category name."
                ),
            },
            {"role": "user", "content": state["query"]},
        ],
        temperature=0,
    )
    cat = resp.choices[0].message.content.strip().lower()
    print(f"  [categorize] -> {cat}")
    return {"category": cat}


def handle_technical(state: RouterState) -> dict:
    print("  [handle_technical] Routing to technical support")
    return {"response": f"Technical support response for: {state['query']}"}


def handle_billing(state: RouterState) -> dict:
    print("  [handle_billing] Routing to billing team")
    return {"response": f"Billing response for: {state['query']}"}


def handle_general(state: RouterState) -> dict:
    print("  [handle_general] Routing to general support")
    return {"response": f"General response for: {state['query']}"}


def route_by_category(state: RouterState) -> str:
    """Routing function: returns the next node name based on category."""
    category = state["category"]
    if category == "technical":
        return "technical"
    elif category == "billing":
        return "billing"
    else:
        return "general"


# Build the conditional graph
router_graph = StateGraph(RouterState)
router_graph.add_node("categorize", categorize)
router_graph.add_node("technical", handle_technical)
router_graph.add_node("billing", handle_billing)
router_graph.add_node("general", handle_general)

router_graph.set_entry_point("categorize")

# Conditional edges: categorize -> one of three handlers
router_graph.add_conditional_edges(
    "categorize",
    route_by_category,
    {
        "technical": "technical",
        "billing": "billing",
        "general": "general",
    },
)

# All handlers lead to END
for node in ["technical", "billing", "general"]:
    router_graph.add_edge(node, END)

router_app = router_graph.compile()

# Test with different queries
for query in [
    "My API key isn't working, I get a 401 error",
    "I was charged twice on my credit card",
    "What are your business hours?",
]:
    print(f"\nQuery: '{query}'")
    result = router_app.invoke({"query": query, "category": "", "response": ""})
    print(f"Response: {result['response']}")

### 3.2 Loops with Conditional Edges

LangGraph supports **cycles** -- looping back to earlier nodes. This is essential
for iterative workflows like the research assistant below, which loops through
research steps before synthesizing results.

In [ ]:
# --- Research assistant with looping ---

class ResearchState(TypedDict):
    task: str
    plan: list[str]
    results: Annotated[list[str], add]  # Accumulates across nodes via the `add` reducer
    current_step: int
    final_answer: str


def planner(state: ResearchState) -> dict:
    """Break task into sub-steps using the LLM."""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "user",
                "content": (
                    f"Break this research task into 2-3 concrete steps. "
                    f"Return as a JSON object with a 'steps' array of strings.\n\n"
                    f"Task: {state['task']}"
                ),
            }
        ],
        response_format={"type": "json_object"},
    )
    steps = json.loads(response.choices[0].message.content)["steps"]
    print(f"  [planner] Created {len(steps)} steps: {steps}")
    return {"plan": steps, "current_step": 0}


def researcher(state: ResearchState) -> dict:
    """Execute the current research step."""
    step = state["plan"][state["current_step"]]
    print(f"  [researcher] Step {state['current_step'] + 1}: {step}")

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": f"Research this: {step}"}],
    )
    result = response.choices[0].message.content
    return {"results": [result], "current_step": state["current_step"] + 1}


def should_continue(state: ResearchState) -> str:
    """Check if more research steps remain."""
    if state["current_step"] < len(state["plan"]):
        return "research"    # Loop back
    return "synthesize"      # Move forward


def synthesizer(state: ResearchState) -> dict:
    """Combine all research results into a final answer."""
    all_results = "\n\n".join(state["results"])
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "user",
                "content": (
                    f"Synthesize these research findings into a concise answer "
                    f"(3-4 sentences).\n\nTask: {state['task']}\n\n"
                    f"Findings:\n{all_results}"
                ),
            }
        ],
    )
    print("  [synthesizer] Produced final answer")
    return {"final_answer": response.choices[0].message.content}


# Build the graph with a loop
research_graph = StateGraph(ResearchState)
research_graph.add_node("planner", planner)
research_graph.add_node("researcher", researcher)
research_graph.add_node("synthesizer", synthesizer)

research_graph.set_entry_point("planner")
research_graph.add_edge("planner", "researcher")

# Conditional edge: loop back or move to synthesis
research_graph.add_conditional_edges(
    "researcher",
    should_continue,
    {
        "research": "researcher",       # Loop back for next step
        "synthesize": "synthesizer",     # All steps done
    },
)
research_graph.add_edge("synthesizer", END)

research_app = research_graph.compile()

# Run
result = research_app.invoke({
    "task": "Compare vLLM and TGI for production LLM deployment",
    "plan": [],
    "results": [],
    "current_step": 0,
    "final_answer": "",
})

print(f"\nFinal Answer:\n{result['final_answer']}")

---
## Section 4: Multi-Agent Systems (Supervisor Pattern)

A single agent with many tools becomes unreliable as complexity grows. **Multi-agent
systems** split responsibilities across specialized agents that each excel at one thing.

Common patterns:

| Pattern | Description | Best For |
|---|---|---|
| **Supervisor** | One manager delegates to specialists | Complex task routing |
| **Sequential** | Agents pass work in a pipeline | Content creation, reports |
| **Collaborative** | Agents debate and refine together | Code generation, reviews |

Below we implement the **supervisor pattern**: a manager agent receives the user
request, decides which specialist to invoke, and collects results.

In [ ]:
# --- Multi-Agent Supervisor Pattern ---

class TeamState(TypedDict):
    task: str
    next_agent: str
    research: str
    code: str
    review: str
    final_output: str


def supervisor(state: TeamState) -> dict:
    """Supervisor decides which specialist agent should work next."""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a team supervisor. Based on the current task state, "
                    "decide which agent should work next:\n"
                    "- 'researcher': Needs information gathering\n"
                    "- 'coder': Needs code written or modified\n"
                    "- 'reviewer': Code is ready for quality review\n"
                    "- 'FINISH': Task is complete\n\n"
                    "Return ONLY the agent name, nothing else."
                ),
            },
            {
                "role": "user",
                "content": (
                    f"Task: {state['task']}\n"
                    f"Research: {state.get('research') or 'Not done'}\n"
                    f"Code: {state.get('code') or 'Not written'}\n"
                    f"Review: {state.get('review') or 'Not reviewed'}"
                ),
            },
        ],
        temperature=0.0,
    )
    next_agent = response.choices[0].message.content.strip().lower()
    print(f"  [supervisor] -> {next_agent}")
    return {"next_agent": next_agent}


def researcher_agent(state: TeamState) -> dict:
    """Specialist: researches technical topics."""
    print("  [researcher] Gathering information...")
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": "You are a technical researcher. Provide concise research findings (3-4 bullet points).",
            },
            {"role": "user", "content": f"Research this topic: {state['task']}"},
        ],
    )
    return {"research": response.choices[0].message.content}


def coder_agent(state: TeamState) -> dict:
    """Specialist: writes production code."""
    print("  [coder] Writing code...")
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": "You are an expert Python developer. Write clean, concise code. Include brief comments.",
            },
            {
                "role": "user",
                "content": f"Task: {state['task']}\nResearch: {state['research']}\nWrite the code.",
            },
        ],
    )
    return {"code": response.choices[0].message.content}


def reviewer_agent(state: TeamState) -> dict:
    """Specialist: reviews code for quality and correctness."""
    print("  [reviewer] Reviewing code...")
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a senior code reviewer. Give a brief review "
                    "(2-3 sentences) noting any bugs or improvements."
                ),
            },
            {"role": "user", "content": f"Review this code:\n{state['code']}"},
        ],
    )
    return {"review": response.choices[0].message.content}


def route_supervisor(state: TeamState) -> str:
    """Route based on the supervisor's decision."""
    agent = state["next_agent"]
    if agent == "finish" or agent == "\"finish\"":
        return END
    return agent


# Build multi-agent graph
team_graph = StateGraph(TeamState)
team_graph.add_node("supervisor", supervisor)
team_graph.add_node("researcher", researcher_agent)
team_graph.add_node("coder", coder_agent)
team_graph.add_node("reviewer", reviewer_agent)

team_graph.set_entry_point("supervisor")

# Supervisor routes to the appropriate specialist
team_graph.add_conditional_edges(
    "supervisor",
    route_supervisor,
    {
        "researcher": "researcher",
        "coder": "coder",
        "reviewer": "reviewer",
        END: END,
    },
)

# All specialists report back to supervisor
for agent_name in ["researcher", "coder", "reviewer"]:
    team_graph.add_edge(agent_name, "supervisor")

team_app = team_graph.compile()

# Run the multi-agent team
print("Running multi-agent team...\n")
team_result = team_app.invoke({
    "task": "Write a Python function that retries HTTP requests with exponential backoff",
    "next_agent": "",
    "research": "",
    "code": "",
    "review": "",
    "final_output": "",
})

print("\n" + "=" * 60)
print("Research:\n", team_result["research"][:300], "...")
print("\nCode:\n", team_result["code"][:500], "...")
print("\nReview:\n", team_result["review"])

---
## Section 5: Human-in-the-Loop with `interrupt_before`

Fully autonomous agents are powerful but risky. An agent that can send emails or
modify databases could cause damage if it makes a wrong decision.

**Human-in-the-loop (HITL)** adds checkpoints where the agent pauses for human
approval before taking high-impact actions.

Design principle: gate actions based on **reversibility** and **blast radius**.

| Action | Gate? |
|---|---|
| Reading data | No gate needed |
| Drafting content | Light review |
| Sending email to 10k users | Hard stop for approval |
| Modifying production DB | Hard stop for approval |

LangGraph supports HITL through its `interrupt_before` / `interrupt_after`
mechanism. The graph saves its complete state and pauses. A human reviews,
approves or rejects, and the graph resumes from exactly where it stopped.

In [ ]:
from langgraph.checkpoint.memory import MemorySaver


class ApprovalState(TypedDict):
    task: str
    proposed_action: str
    approved: bool
    result: str


def plan_action(state: ApprovalState) -> dict:
    """Agent plans an action that requires human approval."""
    action = f"Send refund email to customer regarding: {state['task']}"
    print(f"  [plan_action] Proposed: {action}")
    return {"proposed_action": action}


def execute_action(state: ApprovalState) -> dict:
    """Execute the action (only if approved)."""
    if state["approved"]:
        result = f"Successfully executed: {state['proposed_action']}"
        print(f"  [execute_action] {result}")
    else:
        result = "Action was rejected by human reviewer."
        print(f"  [execute_action] REJECTED")
    return {"result": result}


# Build graph with HITL interrupt
hitl_graph = StateGraph(ApprovalState)
hitl_graph.add_node("plan", plan_action)
hitl_graph.add_node("execute", execute_action)

hitl_graph.set_entry_point("plan")
hitl_graph.add_edge("plan", "execute")
hitl_graph.add_edge("execute", END)

# Compile with checkpointer and interrupt BEFORE the execute node
checkpointer = MemorySaver()
hitl_app = hitl_graph.compile(
    checkpointer=checkpointer,
    interrupt_before=["execute"],  # Pause here for human approval
)

# --- Phase 1: Run until the interrupt ---
config = {"configurable": {"thread_id": "approval-demo-1"}}

print("Phase 1: Running until interrupt...\n")
state = hitl_app.invoke(
    {
        "task": "refund order #12345",
        "proposed_action": "",
        "approved": False,
        "result": "",
    },
    config=config,
)

print(f"\nGraph paused. Proposed action: '{state['proposed_action']}'")
print("Waiting for human approval...")

In [ ]:
# --- Phase 2: Human approves and resumes ---
print("Phase 2: Human approves the action.\n")

# Update the state with approval
hitl_app.update_state(config, {"approved": True})

# Resume execution from the checkpoint
final_state = hitl_app.invoke(None, config=config)

print(f"\nFinal result: {final_state['result']}")

---
## Section 6: Agent Memory and State Persistence

LangGraph's **checkpointing** system enables:

- **Pause and resume** -- the graph can stop (e.g., for HITL) and restart later
- **Thread-based memory** -- each conversation thread maintains its own state
- **Fault tolerance** -- if the server crashes, the graph resumes from the last checkpoint

The `MemorySaver` stores state in memory (suitable for development). For production,
use `SqliteSaver` or `PostgresSaver` for persistent storage.

In [ ]:
# --- Demonstrating state persistence with threads ---

class ConversationState(TypedDict):
    messages: Annotated[list[str], add]
    turn_count: int


def chat_node(state: ConversationState) -> dict:
    """Process the latest message."""
    latest = state["messages"][-1]
    reply = f"Agent reply to: '{latest}' (turn {state['turn_count'] + 1})"
    return {
        "messages": [reply],
        "turn_count": state["turn_count"] + 1,
    }


memory_graph = StateGraph(ConversationState)
memory_graph.add_node("chat", chat_node)
memory_graph.set_entry_point("chat")
memory_graph.add_edge("chat", END)

memory_saver = MemorySaver()
memory_app = memory_graph.compile(checkpointer=memory_saver)

# Thread 1: Multiple turns maintain state
thread_config = {"configurable": {"thread_id": "user-alice"}}

for msg in ["Hello!", "Tell me about agents", "What is LangGraph?"]:
    result = memory_app.invoke(
        {"messages": [msg], "turn_count": 0},
        config=thread_config,
    )
    print(f"Turn {result['turn_count']}: {result['messages'][-1]}")

# Inspect the persisted state
snapshot = memory_app.get_state(thread_config)
print(f"\nPersisted state for thread 'user-alice':")
print(f"  Total messages: {len(snapshot.values['messages'])}")
print(f"  Turn count: {snapshot.values['turn_count']}")

---
## Section 7: Error Handling in Agent Workflows

Agents introduce unique safety challenges:
- A chatbot that generates wrong text is annoying.
- An agent that *executes* wrong actions can delete data, send bad emails, or burn cloud budgets.

Key safety layers:

| Safety Layer | What It Prevents | Implementation |
|---|---|---|
| Max iterations | Infinite loops | Counter in agent loop |
| Token budget | Runaway costs | Token counter + hard limit |
| Tool timeouts | Hung tool calls | Timeout wrapper |
| Input validation | Injection attacks | Pydantic schemas |
| Audit logging | Untrackable actions | Structured logs per step |
| Human approval | Irreversible actions | `interrupt_before` |

In [ ]:
# --- Safety Wrapper for Tools ---
logger = logging.getLogger("agent")
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")


def safe_tool(max_retries: int = 2, timeout_seconds: int = 30):
    """Decorator that adds retry logic, timeout tracking, and logging to agent tools."""

    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(max_retries + 1):
                start = time.perf_counter()
                try:
                    result = func(*args, **kwargs)
                    elapsed = time.perf_counter() - start

                    logger.info(
                        f"Tool={func.__name__} status=success "
                        f"elapsed={elapsed:.2f}s attempt={attempt + 1}"
                    )

                    if elapsed > timeout_seconds:
                        logger.warning(
                            f"Tool {func.__name__} exceeded soft timeout "
                            f"({elapsed:.1f}s > {timeout_seconds}s)"
                        )

                    return result

                except Exception as e:
                    logger.error(
                        f"Tool={func.__name__} error={e} attempt={attempt + 1}"
                    )
                    if attempt == max_retries:
                        return f"Error after {max_retries + 1} attempts: {e}"

        return wrapper

    return decorator


# --- Cost/Budget Tracking ---
class AgentBudget:
    """Track and limit agent spending on LLM calls."""

    def __init__(self, max_llm_calls: int = 20, max_tokens: int = 50_000):
        self.max_llm_calls = max_llm_calls
        self.max_tokens = max_tokens
        self.llm_calls = 0
        self.total_tokens = 0

    def track(self, tokens_used: int):
        """Record an LLM call. Raises RuntimeError if budget exceeded."""
        self.llm_calls += 1
        self.total_tokens += tokens_used
        if self.llm_calls > self.max_llm_calls:
            raise RuntimeError(
                f"Agent exceeded max LLM calls: {self.max_llm_calls}"
            )
        if self.total_tokens > self.max_tokens:
            raise RuntimeError(
                f"Agent exceeded token budget: {self.max_tokens}"
            )

    def summary(self) -> str:
        return (
            f"LLM calls: {self.llm_calls}/{self.max_llm_calls}, "
            f"Tokens: {self.total_tokens:,}/{self.max_tokens:,}"
        )


# Demo: safe_tool decorator
@safe_tool(max_retries=2, timeout_seconds=5)
def flaky_tool(x: int) -> str:
    """A tool that sometimes fails (for demonstration)."""
    import random

    if random.random() < 0.5:
        raise ConnectionError("Simulated network failure")
    return f"Success: processed {x}"


print("Testing flaky_tool with retries:")
for i in range(3):
    result = flaky_tool(i)
    print(f"  Call {i}: {result}")

# Demo: budget tracker
budget = AgentBudget(max_llm_calls=5, max_tokens=10_000)
for i in range(4):
    budget.track(tokens_used=2000)
    print(f"  After call {i + 1}: {budget.summary()}")

try:
    budget.track(tokens_used=3000)
except RuntimeError as e:
    print(f"  Budget exceeded: {e}")

---
## Section 8: End-to-End Agent -- Research Assistant

This section brings everything together into a complete research assistant agent
that:

1. **Plans** research steps from a user query
2. **Researches** each step (with tools and LLM)
3. **Loops** until all steps are complete
4. **Reviews** the quality of findings
5. **Synthesizes** a final report
6. Includes **checkpointing**, **budget tracking**, and **error handling**

This combines: ReAct pattern, LangGraph state machine, conditional routing,
memory/persistence, and safety guardrails.

In [ ]:
# --- End-to-End Research Assistant Agent ---

class ResearchAssistantState(TypedDict):
    """Full state for the research assistant."""
    # Input
    query: str

    # Planning
    research_plan: list[str]
    current_step_index: int

    # Research
    findings: Annotated[list[str], add]

    # Quality check
    quality_score: str  # 'pass' or 'fail'
    revision_count: int

    # Output
    final_report: str
    status: str  # 'planning', 'researching', 'reviewing', 'done', 'error'


# --- Budget tracker (shared across nodes) ---
research_budget = AgentBudget(max_llm_calls=15, max_tokens=40_000)


def tracked_llm_call(messages, model="gpt-4o-mini", **kwargs):
    """Wrapper that tracks token usage for budget enforcement."""
    response = client.chat.completions.create(
        model=model, messages=messages, **kwargs
    )
    tokens_used = response.usage.total_tokens if response.usage else 500
    research_budget.track(tokens_used)
    return response


# --- Node: Plan ---
def plan_research(state: ResearchAssistantState) -> dict:
    """Break the query into concrete research steps."""
    print(f"  [plan] Planning research for: {state['query']}")
    response = tracked_llm_call(
        messages=[
            {
                "role": "user",
                "content": (
                    f"Break this research question into 2-3 specific, actionable "
                    f"research steps. Return as JSON with a 'steps' array.\n\n"
                    f"Question: {state['query']}"
                ),
            }
        ],
        response_format={"type": "json_object"},
    )
    steps = json.loads(response.choices[0].message.content).get("steps", [])
    print(f"  [plan] {len(steps)} steps created")
    return {
        "research_plan": steps,
        "current_step_index": 0,
        "status": "researching",
    }


# --- Node: Research ---
def do_research(state: ResearchAssistantState) -> dict:
    """Execute the current research step."""
    idx = state["current_step_index"]
    step = state["research_plan"][idx]
    print(f"  [research] Step {idx + 1}/{len(state['research_plan'])}: {step}")

    response = tracked_llm_call(
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a thorough researcher. Provide detailed, factual "
                    "findings for the given research step. Be specific and "
                    "cite concepts where relevant."
                ),
            },
            {"role": "user", "content": step},
        ],
    )
    finding = response.choices[0].message.content
    return {
        "findings": [f"## Step {idx + 1}: {step}\n{finding}"],
        "current_step_index": idx + 1,
    }


# --- Routing: Continue researching or move to review ---
def route_after_research(state: ResearchAssistantState) -> str:
    if state["current_step_index"] < len(state["research_plan"]):
        return "research"  # Loop back
    return "review"  # All steps done


# --- Node: Quality Review ---
def review_quality(state: ResearchAssistantState) -> dict:
    """Check if findings are sufficient to answer the original query."""
    all_findings = "\n\n".join(state["findings"])
    print(f"  [review] Checking quality (revision #{state.get('revision_count', 0)})")

    response = tracked_llm_call(
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a research quality reviewer. Evaluate whether "
                    "the findings adequately answer the original question. "
                    "Return JSON with 'quality': 'pass' or 'fail' and "
                    "a brief 'reason'."
                ),
            },
            {
                "role": "user",
                "content": (
                    f"Original question: {state['query']}\n\n"
                    f"Findings:\n{all_findings}"
                ),
            },
        ],
        response_format={"type": "json_object"},
    )
    result = json.loads(response.choices[0].message.content)
    quality = result.get("quality", "pass")
    reason = result.get("reason", "")
    print(f"  [review] Quality: {quality} -- {reason}")
    return {
        "quality_score": quality,
        "revision_count": state.get("revision_count", 0) + 1,
    }


# --- Routing: Pass -> synthesize, Fail -> research again (max 1 retry) ---
def route_after_review(state: ResearchAssistantState) -> str:
    if state["quality_score"] == "pass" or state["revision_count"] >= 2:
        return "synthesize"
    # Reset step index to re-research
    return "research"


# --- Node: Synthesize ---
def synthesize_report(state: ResearchAssistantState) -> dict:
    """Combine all findings into a polished final report."""
    all_findings = "\n\n".join(state["findings"])
    print("  [synthesize] Generating final report...")

    response = tracked_llm_call(
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a research report writer. Synthesize the findings "
                    "into a clear, well-structured report (4-6 paragraphs). "
                    "Include a brief conclusion with key takeaways."
                ),
            },
            {
                "role": "user",
                "content": (
                    f"Question: {state['query']}\n\n"
                    f"Findings:\n{all_findings}"
                ),
            },
        ],
    )
    return {
        "final_report": response.choices[0].message.content,
        "status": "done",
    }


print("All node functions defined.")

In [ ]:
# --- Build the full research assistant graph ---

ra_graph = StateGraph(ResearchAssistantState)

# Add nodes
ra_graph.add_node("plan", plan_research)
ra_graph.add_node("research", do_research)
ra_graph.add_node("review", review_quality)
ra_graph.add_node("synthesize", synthesize_report)

# Set entry point
ra_graph.set_entry_point("plan")

# Edges
ra_graph.add_edge("plan", "research")

# Conditional: after research, loop or review
ra_graph.add_conditional_edges(
    "research",
    route_after_research,
    {"research": "research", "review": "review"},
)

# Conditional: after review, re-research or synthesize
ra_graph.add_conditional_edges(
    "review",
    route_after_review,
    {"research": "research", "synthesize": "synthesize"},
)

ra_graph.add_edge("synthesize", END)

# Compile with checkpointing
ra_memory = MemorySaver()
research_assistant = ra_graph.compile(checkpointer=ra_memory)

print("Research assistant graph compiled.")
print(f"Nodes: {list(research_assistant.get_graph().nodes.keys())}")

In [ ]:
# --- Run the research assistant ---

# Reset budget for this run
research_budget = AgentBudget(max_llm_calls=15, max_tokens=40_000)

ra_config = {"configurable": {"thread_id": "research-demo-1"}}

print("Running research assistant...\n")
print("=" * 60)

ra_result = research_assistant.invoke(
    {
        "query": "What are the key differences between LangGraph and CrewAI for building multi-agent systems?",
        "research_plan": [],
        "current_step_index": 0,
        "findings": [],
        "quality_score": "",
        "revision_count": 0,
        "final_report": "",
        "status": "planning",
    },
    config=ra_config,
)

print("\n" + "=" * 60)
print(f"Status: {ra_result['status']}")
print(f"Budget: {research_budget.summary()}")
print(f"Steps completed: {ra_result['current_step_index']}")
print(f"Quality: {ra_result['quality_score']}")
print(f"Revisions: {ra_result['revision_count']}")
print(f"\n{'=' * 60}")
print("FINAL REPORT:")
print("=" * 60)
print(ra_result["final_report"])

---
## Summary

In this notebook we covered the complete landscape of LLM agents:

1. **ReAct Pattern** -- Built an agent loop from scratch: Reason, Act, Observe, Repeat
2. **Tool Definition & Binding** -- JSON schemas, `@tool` decorator, `bind_tools()`
3. **LangGraph Basics** -- `StateGraph`, nodes, edges, `TypedDict` state
4. **Conditional Routing** -- Branching and looping with `add_conditional_edges()`
5. **Multi-Agent Supervisor** -- Manager delegates to researcher, coder, reviewer
6. **Human-in-the-Loop** -- `interrupt_before` for approval gates, `update_state()` to resume
7. **Memory & Persistence** -- `MemorySaver` checkpointer, thread-based state
8. **Safety & Reliability** -- Retry decorators, budget tracking, structured logging
9. **End-to-End Agent** -- Research assistant with planning, looping, quality review, and synthesis

**Key design principles:**
- Use the simplest agent architecture that solves your problem
- Gate dangerous actions with human-in-the-loop approval
- Always cap iterations and token budgets to prevent runaway costs
- Log every tool call and decision for observability and debugging

**Next module**: `10-evaluation.html` -- How to evaluate LLM and agent outputs systematically.